In [ ]:
%load_ext autoreload
%autoreload 2

**Author:** Salvador Navas  
**Date:** 2025-06-27

In [ ]:
import os
from pathlib import Path

from pyhydra.data_sources.rainfall import download_aemet_daily_data
from pyhydra.data_sources.rainfall import AEMETDownloader, AemetCSVLoader

# Portable repo-root / data-dir resolution (local clone, Docker, Azure ML)
_cwd = Path.cwd()
_candidates = [Path('/workspace'), _cwd, *_cwd.parents]
REPO_ROOT = next(
    (p for p in _candidates if (p / 'notebooks').exists() or (p / 'pyhydra').exists()),
    _cwd,
)
DATA_DIR = Path(os.environ.get('HYDRA_DATA_DIR', str(REPO_ROOT / 'data')))


# 🌦️ AEMET Daily Climate Data Downloader

## General Description

This tool downloads **daily meteorological observations** from the **AEMET OpenData API**
(Agencia Estatal de Meteorología, Spain).

#### 🆚 AEMET vs alternatives — when to use which?

| Source | Coverage | Stations | Quality | Best for |
|--------|----------|----------|---------|----------|
| **AEMET** | Spain only | ~900 (dense) | High (national QC) | Spain studies, long records, official data |
| **Meteostat** | Global | ~70k (sparse) | Moderate | Quick downloads, international stations |
| **OGIMET** | Global | WMO network | Variable | Sub-daily SYNOP, cloud/visibility |
| **ERA5** | Global | 0.25° grid | Reanalysis | Ungauged areas, spatial fields |

> **For Spain**: AEMET is the reference source. It provides the densest network,
> longest records (some stations back to 1900s), and official national quality control.

## 📋 Variables available

| Column | Description | Unit |
|--------|-------------|------|
| `prec` | Daily precipitation | mm |
| `tmax` | Daily maximum temperature | °C |
| `tmin` | Daily minimum temperature | °C |
| `tmed` | Daily mean temperature | °C |
| `velmedia` | Mean wind speed | m/s |
| `racha` | Maximum wind gust | m/s |
| `sol` | Sunshine duration | h |
| `presMax` / `presMin` | Max/min atmospheric pressure | hPa |
| `hrMedia` | Mean relative humidity | % |

## 🔑 Prerequisites

Register at https://opendata.aemet.es and obtain a free API key. Set `API_KEY` below.

## 🗺️ Workflow — two modes (auto-detected)

| Mode | When | Workflow |
|------|------|----------|
| **Direct** *(recommended)* | API key set, no pre-downloaded data | Inventory fetched from API → select stations on map → download only chosen stations to CSV |
| **Bulk** | NetCDF files already present in `data/aemet/` | Select on map → extract from pre-downloaded files |

> **Direct mode** is the default for most users: set your API key, open the widget, draw a
> rectangle on the map or click individual stations, and download — no multi-GB pre-download needed.
>
> **Bulk mode** (via `download_aemet_daily_data`) downloads all ~900 stations at once (30–60 min,
> several GB). Use it only for full-network spatial analyses.

## 🗂️ Station reference data

The file `data/aemet/Estaciones_Auto_AEMET_IHC.shp` contains the locations of all automatic
AEMET stations as a GIS shapefile (EPSG:4258). It is included in the repo as a static reference.
The interactive widget fetches the full inventory with station IDs directly from the AEMET API.

In [ ]:
# === CONFIGURATION ===
API_KEY    = 'YOUR_AEMET_API_KEY'   # Get yours at https://opendata.aemet.es
OUTPUT_DIR = str(DATA_DIR / 'aemet')

if API_KEY == 'YOUR_AEMET_API_KEY':
    print('[AEMET] API key not configured — set API_KEY above.')
    print('[AEMET] Get your key at: https://opendata.aemet.es')

# 📊 Interactive Station Selector

The **AEMETDownloader widget** works in two modes (auto-detected):

**Direct mode** *(default — API key required)*
- Fetches the full station inventory from AEMET (~900 stations, < 1 s)
- Displays all stations on the interactive map
- Draw a rectangle or click individual markers to select stations
- Click **Download Selected Stations** → only the chosen stations are downloaded to CSV

**Bulk mode** *(when NetCDF files are present in `OUTPUT_DIR`)*
- Uses pre-downloaded files from `download_aemet_daily_data()`
- Same map interface — extracts the selected stations from the local NetCDF cache

In both modes, the widget saves:
- `AEMET_{idema}_series.csv` — one file per station
- `selected_stations.csv` — metadata of the downloaded stations (idema, nombre, lat, lon, alt, provincia)

---
### Optional: bulk download (for full-network spatial analyses only)

```python
download_aemet_daily_data(
    path_output=OUTPUT_DIR,
    api_key=API_KEY,
    start_date_str='2010-01-01T00:00:00UTC',
    end_date_str='2024-12-31T23:59:59UTC',
)
```
> ⚠️ Downloads all ~900 stations — takes 30–60 min and uses several GB of disk space.

In [ ]:
AEMETDownloader(OUTPUT_DIR, api_key=API_KEY)

In [ ]:
loader = AemetCSVLoader(str(DATA_DIR / 'aemet'))


In [ ]:
loader = AemetCSVLoader(OUTPUT_DIR)

In [ ]:
try:
    series_df = loader.load_series_data('prec')
    series_df.head()
except Exception as e:
    print(f'[AEMET] Skipped (no data available): {e}')


[AEMET] Skipped (no data available): [Errno 2] No such file or directory: '/workspace/data/aemet/'


In [ ]:
try:
    quality_df = loader.analyze_series_quality()
    quality_df
except Exception as e:
    print(f'[AEMET] Skipped (no data available): {e}')


[AEMET] Skipped (no data available): Load data first with load_series_data().


---
## 📋 Quality metrics interpretation

`analyze_series_quality()` checks each downloaded station series:

| Column | Meaning | Threshold |
|--------|---------|----------|
| `missing_percent` | % of days with NaN | >5%: infill; >10%: exclude from extremes |
| `full_years` | Years with <1% missing | Need ≥10 for frequency analysis |
| `min_value` / `max_value` | Range check | Negative precipitation → data error |

#### ⚠️ AEMET-specific quality notes

- Some old stations have **instrument changes** that cause sudden shifts in mean values —
  look for step changes in the annual means plot
- Precipitation records marked `'Ip'` (inapreciable) represent trace amounts (<0.1 mm);
  they are converted to 0.0 in pyhydra
- Mountain stations (>1500 m) may have winter gaps due to snow covering the rain gauge

After quality control, series connect to:
- **Spatial interpolation** → `interpolation` notebook
- **Extreme value analysis** → `extreme_value_analysis` notebook
- **Stochastic generation** → `stochastic_generation` notebook
